# Stability vs. Flexibility — Coding-Assignment Sandbox (A1 + A2)

A hands-on, **fill-in-the-blank** playground for the first two coding assignments in
[`docs/stability_flexibility_coding_assignments.md`](../../../docs/stability_flexibility_coding_assignments.md):

| # | Assignment | Skeleton | You build |
|---|---|---|---|
| **A1** | Per-electrode two-way **ANOVA** electrode definition | `docs/skeletons/a1_anova_labels.py` | `_anova_interaction_stats`, `per_electrode_anova_labels` |
| **A2** | Conjunction **permutation null** + **threshold sweep** | `docs/skeletons/a2_conjunction_null_sweep.py` | `conjunction_permutation_null`, `conjunction_threshold_sweep` |

Both drop into `src/analysis/stats/stability_flexibility_segregation.py` when finished.

### How this notebook works
1. **Setup** loads real iEEG data for **1–2 subjects, clipped to LPFC electrodes** — and
   automatically **falls back to a synthetic ground-truth dataset** if the real data isn't
   reachable (e.g. you're off the cluster / Box). Everything downstream runs either way.
2. Each **Task** gives you a stub cell that raises `NotImplementedError`. Fill it in.
3. Stuck? **Reveal hints on demand** — no scrolling past spoilers:
   - `reveal("a1_stats_hint1")` … tiered nudges: *what to reach for* (libraries/functions).
   - `reveal("a1_stats_solution")` … the reference implementation, to read/copy.
   - `load_reference("a1_stats")` … inject the reference into the namespace so you can keep
     going even if you'd rather not implement that piece yet.
4. Each task ends with an **acceptance-check** cell that encodes the assignment's success
   criteria (agreement with the nonparametric labels, near-null cross-controls, invariant
   permutations, OR≈1 on independent data).

> The `reveal(...)` and `load_reference(...)` helpers are defined in the setup section.
> Hints/solutions live in a registry so they stay out of sight until you ask.

---
## 0 · Setup — imports, paths, reveal helpers

In [ ]:
# --- path setup so `src...` and `dcc_scripts...` import cleanly ---------------
import os, sys, textwrap, warnings
warnings.simplefilter("ignore")

# Walk up to the repo root (the folder containing `src/` and `docs/`).
_here = os.getcwd()
_root = _here
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("repo root:", _root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# statsmodels pieces you'll use in A1
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

# helpers already in the segregation module — reuse these, don't rebuild them
from src.analysis.stats.stability_flexibility_segregation import (
    _canonical_labels, _interaction_effect, _is_interaction,
    resolve_contrasts, finalize_contrasts,
    per_electrode_labels, cmh_conjunction, subject_clustered_corr,
    compute_sensitivities, add_responsiveness, prepare_continuous,
    _synthetic_df,
)
print("segregation module imported OK")

### The reveal registry
Run this once. It only *stores* hints and solutions — nothing is shown until you call
`reveal(...)`. (Peeking here is allowed, but it's the whole answer key.)

In [ ]:

from IPython.display import display, Markdown, Code

_REVEALS = {}   # key -> ("md" | "code", text)

def _register(key, kind, text):
    _REVEALS[key] = (kind, textwrap.dedent(text).strip("\n"))

def reveal(key):
    """Show a hint (markdown) or a reference solution (code) on demand."""
    if key not in _REVEALS:
        print(f"no reveal named {key!r}. Available:")
        for k in sorted(_REVEALS): print("   ", k)
        return
    kind, text = _REVEALS[key]
    if kind == "code":
        display(Markdown(f"**Reference solution — `{key}`**  "
                         f"(read/copy above, or run `load_reference('{key}')`):"))
        display(Code(text, language="python"))
    else:
        display(Markdown(text))

def load_reference(key):
    """Exec a reference solution into the notebook namespace so you can proceed.
    Accepts the short key ('a1_stats') or the full one ('a1_stats_solution')."""
    resolved = key if key in _REVEALS else f"{key}_solution"
    kind, text = _REVEALS.get(resolved, (None, None))
    if kind != "code":
        raise KeyError(f"{key!r} is not a loadable code reference "
                       f"(try one of: {[k for k in _REVEALS if k.endswith('_solution')]})")
    exec(compile(text, f"<reference:{resolved}>", "exec"), globals())
    print(f"✓ loaded reference implementation: {resolved}")

def list_reveals(prefix=""):
    for k in sorted(_REVEALS):
        if k.startswith(prefix):
            print(f"  {k:28s} [{_REVEALS[k][0]}]")


# ------------------------------- A1 hints/solutions -------------------------
_register("a1_stats_hint1", "md", """
**A1 · `_anova_interaction_stats` — hint 1 (the trap).**
This is a per-electrode *two-way* ANOVA, but the point of the assignment is a
**leakage trap**: the proportion cells are deliberately unequal (~75/25). With
statsmodels' default **treatment** coding, the Type III interaction sum-of-squares is
*not* orthogonal to the main effects, so a pure congruency/switch **main effect leaks
into the "interaction"**. Fix it with **Type III SS + sum/effect coding** — the
parametric analogue of the equal-cell-weight difference-of-differences the module
already uses in `_interaction_effect`.
""")

_register("a1_stats_hint2", "md", """
**A1 · `_anova_interaction_stats` — hint 2 (what to reach for).**
- `statsmodels.formula.api.ols` for the fit; `statsmodels.stats.anova.anova_lm(model, typ=3)`.
- Sum-code **both** factors in the formula so Type III is orthogonal:
  `f"{hg_col} ~ C({cond_col}, Sum) * C({mod_col}, Sum)"`.
- The interaction row's index is literally `f"C({cond_col}, Sum):C({mod_col}, Sum)"`;
  read `'F'` and `'PR(>F)'` from it.
- Wrap the fit in `try/except` and return `{'F': np.nan, 'p': np.nan}` on a singular
  design (an electrode missing a cell) — mirror how the module's effect helpers return
  NaN for too-few trials.
""")

_register("a1_stats_solution", "code", """
def _anova_interaction_stats(elec_df, cond_col, mod_col, hg_col="hg"):
    \"\"\"One electrode's two-way Type III ANOVA; returns the interaction F and p.\"\"\"
    try:
        formula = f"{hg_col} ~ C({cond_col}, Sum) * C({mod_col}, Sum)"
        model = smf.ols(formula, data=elec_df).fit()
        aov = anova_lm(model, typ=3)                       # Type III
        inter = f"C({cond_col}, Sum):C({mod_col}, Sum)"
        return {"F": float(aov.loc[inter, "F"]),
                "p": float(aov.loc[inter, "PR(>F)"])}
    except Exception:
        return {"F": np.nan, "p": np.nan}                  # singular / missing cell
""")

_register("a1_labels_hint1", "md", """
**A1 · `per_electrode_anova_labels` — hint 1 (scaffold + sign).**
- Canonicalize once up front so the `_scond/_smod/_fcond/_fmod` cell labels exist:
  `contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))`
  then `work = _canonical_labels(df, contrasts)`.
- Loop `work.groupby(["subject", "electrode"])`; call `_anova_interaction_stats`
  twice: stability = congruency × incongruent_proportion, flexibility =
  switchType × switch_proportion.
- The F-test is **unsigned**. Get the sign from the module's *own* estimator so it
  matches the §2 correlation exactly:
  `np.sign(_interaction_effect(hg, cond, mod, "cohens_d", alpha))` using
  `_scond/_smod` (stability) and `_fcond/_fmod` (flexibility).
""")

_register("a1_labels_hint2", "md", """
**A1 · `per_electrode_anova_labels` — hint 2 (FDR, flags, cross-controls).**
- FDR across electrodes: `q = multipletests(p.fillna(1), method="fdr_bh")[1]` for
  `p_cong` and `p_switch` separately.
- Flags: `S = (q_cong < alpha)`, `F = (q_switch < alpha)`. You *may* additionally
  require the sign to be positive (predicted direction) — decide & document. This
  reference exposes it via `require_sign` (default `False`, to match the
  nonparametric sibling `per_electrode_labels`).
- **Cross controls (report only, never select on):** congruency × switch_proportion
  and switchType × incongruent_proportion. On the synthetic generator there is no
  true cross-effect, so their p-values should be ~uniform.
""")

_register("a1_labels_solution", "code", """
def per_electrode_anova_labels(df, alpha=0.05, contrast_mode="proportion",
                               contrasts=None, include_cross_controls=True,
                               require_sign=False):
    \"\"\"Parametric per-electrode S/F labels from the two-way interaction ANOVA.
    Drop-in `labels` for cmh_conjunction (columns match per_electrode_labels).\"\"\"
    contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))
    work = _canonical_labels(df, contrasts)               # attaches _scond/_smod/...
    recs = []
    for (subj, elec), g in work.groupby(["subject", "electrode"]):
        st = _anova_interaction_stats(g, "congruency", "incongruent_proportion")
        fl = _anova_interaction_stats(g, "switchType", "switch_proportion")
        s_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_scond"].to_numpy(),
                                             g["_smod"].to_numpy(), "cohens_d", alpha))
        f_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_fcond"].to_numpy(),
                                             g["_fmod"].to_numpy(), "cohens_d", alpha))
        rec = dict(subject=subj, electrode=elec,
                   p_cong=st["p"], F_cong=st["F"], s_sign=s_sign,
                   p_switch=fl["p"], F_switch=fl["F"], f_sign=f_sign)
        if include_cross_controls:                        # specificity controls
            rec["p_cross_cs"] = _anova_interaction_stats(
                g, "congruency", "switch_proportion")["p"]
            rec["p_cross_si"] = _anova_interaction_stats(
                g, "switchType", "incongruent_proportion")["p"]
        recs.append(rec)
    out = pd.DataFrame(recs)
    out["q_cong"]   = multipletests(out["p_cong"].fillna(1),   method="fdr_bh")[1]
    out["q_switch"] = multipletests(out["p_switch"].fillna(1), method="fdr_bh")[1]
    S = out["q_cong"] < alpha
    F = out["q_switch"] < alpha
    if require_sign:
        S = S & (out["s_sign"] > 0)
        F = F & (out["f_sign"] > 0)
    out["S"] = S.astype(int)
    out["F"] = F.astype(int)
    return out
""")

# ------------------------------- A2 hints/solutions -------------------------
_register("a2_null_hint1", "md", """
**A2 · `conjunction_permutation_null` — hint 1 (what to shuffle).**
Shuffle **F within each subject** (leave S untouched). That holds every subject's
S-count and F-count fixed and randomizes only the *pairing* — the exact null CMH
assumes, and it mirrors the within-subject permutation in `subject_clustered_corr`.
Global shuffling would break the subject nesting and manufacture significance.
`observed = int(((S==1) & (F==1)).sum())`.
""")

_register("a2_null_hint2", "md", """
**A2 · `conjunction_permutation_null` — hint 2 (mechanics + the invariant).**
- Precompute per-subject row-index groups (`[np.where(subj==s)[0] for s in unique]`)
  and, each permutation, set `Fp[idx] = F[rng.permutation(idx)]`.
- Two-sided p vs the null **mean**: `(sum(|null-mean| >= |obs-mean|) + 1)/(n_perm+1)`.
- **Acceptance invariant:** every permutation must leave each subject's `S.sum()` and
  `F.sum()` unchanged — only the pairing moves. The acceptance cell asserts this.
""")

_register("a2_null_solution", "code", """
def conjunction_permutation_null(labels, n_perm=10000, seed=0):
    \"\"\"Within-subject permutation null for the count of 'both' (S==1 & F==1) electrodes.\"\"\"
    lab = labels.dropna(subset=["S", "F"]).copy()
    S = lab["S"].to_numpy().astype(int)
    F = lab["F"].to_numpy().astype(int)
    subj = lab["subject"].to_numpy()
    groups = [np.where(subj == s)[0] for s in np.unique(subj)]
    observed = int(((S == 1) & (F == 1)).sum())
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        Fp = F.copy()
        for idx in groups:                 # shuffle F WITHIN each subject only
            Fp[idx] = F[rng.permutation(idx)]
        null[i] = int(((S == 1) & (Fp == 1)).sum())
    mean = null.mean()
    p = (np.sum(np.abs(null - mean) >= abs(observed - mean)) + 1) / (n_perm + 1)
    z = (observed - mean) / null.std() if null.std() > 0 else np.nan
    return dict(observed=observed, null=null, p_two_sided=float(p), z=float(z))
""")

_register("a2_sweep_hint1", "md", """
**A2 · `conjunction_threshold_sweep` — hint 1 (reuse, don't rebuild).**
The sweep is a *loop*: for each threshold, get a fresh labels table from the
`labels_by_threshold` callable, then call the existing `cmh_conjunction(labels)`.
Record `n_S`, `n_F`, `n_both`, `mh_odds_ratio`, and `res['cmh'].pvalue`. Keeping the
callable makes the function agnostic to whether you threshold on q-values (ANOVA /
permutation) or on effect-size percentiles.
""")

_register("a2_sweep_solution", "code", """
def conjunction_threshold_sweep(labels_by_threshold, thresholds):
    \"\"\"Recompute overlap OR + counts across selection thresholds -> tidy DataFrame.\"\"\"
    rows = []
    for t in thresholds:
        lab = labels_by_threshold(t)
        res = cmh_conjunction(lab)
        rows.append(dict(threshold=t,
                         n_S=int(lab.S.sum()), n_F=int(lab.F.sum()),
                         n_both=int(((lab.S == 1) & (lab.F == 1)).sum()),
                         mh_odds_ratio=res["mh_odds_ratio"],
                         cmh_p=res["cmh"].pvalue))
    return pd.DataFrame(rows)
""")

print("reveal registry loaded:")
list_reveals()


### Configuration — point this at your data
Edit the config below to match **your** environment. The loader tries the real epoched
HG data (clipped to LPFC) and **falls back to synthetic** if anything is missing, so the
rest of the notebook always runs.

- `LAB_root=None` auto-resolves the CoganLab root (Box on macOS/Windows, `/cwork/$USER`
  on the DCC). Set it explicitly to override.
- `EPOCHS_ROOT_FILE` is required for a real load — the same value the DCC submit script
  uses (e.g. `Stimulus_1sec_preStimulusBase_decFactor_10`). Left `None` → synthetic.
- `SUBJECTS` = your **one or two** subjects. LPFC electrodes are selected via the
  `lpfc` ROI in `src/analysis/config/rois.py`, using the same machinery as the pipeline.

In [ ]:
from types import SimpleNamespace

CONFIG = SimpleNamespace(
    LAB_root        = None,                       # None -> auto-resolve
    task            = "GlobalLocal",
    epochs_root_file= None,                       # e.g. "Stimulus_1sec_preStimulusBase_decFactor_10"
    subjects        = ["D0057", "D0059"],         # your 1-2 subjects
    window_tmin     = 0.0,                         # analysis window (s post-stimulus)
    window_tmax     = 0.5,
    acc_trials_only = True,
    # synthetic fallback knobs (used only if the real load fails):
    synth_n_subjects= 12,                          # enough subjects for A2 to be non-degenerate
    synth_seed      = 0,
)
CONFIG

#### The loader: real LPFC data → synthetic fallback
`load_sandbox_df(CONFIG)` returns the long-format single-trial table the analysis expects
(one row per electrode×trial): `subject, electrode, hg, congruency, switchType,
incongruent_proportion, switch_proportion`. It uses the **exact** production path
(`load_HG_ev1_rescaled_per_subject` → `resolve_electrodes_to_keep` on the `lpfc` ROI →
`assemble_long_df`). If that path raises (no Box/cluster, missing recon, etc.), it prints
why and returns a 12-subject synthetic table carrying the same columns and a real LWPC/LWPS
interaction, so A1/A2 have recoverable signal.

In [ ]:
def load_real_lpfc_df(cfg):
    """Production path: load 1-2 subjects, clip to LPFC, assemble the long df."""
    from src.analysis.utils.general_utils import (
        resolve_lab_root, resolve_electrodes_to_keep,
        load_HG_ev1_rescaled_per_subject,
    )
    from src.analysis.config.rois import rois_dict
    from dcc_scripts.stats.stability_flexibility_segregation_dcc import assemble_long_df

    if cfg.epochs_root_file is None:
        raise RuntimeError("CONFIG.epochs_root_file is None — set it for a real load.")

    LAB_root = resolve_lab_root(cfg.LAB_root)
    subjects_epochs = load_HG_ev1_rescaled_per_subject(
        subjects=cfg.subjects, epochs_root_file=cfg.epochs_root_file,
        task=cfg.task, LAB_root=LAB_root, acc_trials_only=cfg.acc_trials_only)

    # LPFC clip: reuse resolve_electrodes_to_keep with just the lpfc ROI.
    args = SimpleNamespace(subjects=cfg.subjects, task=cfg.task,
                           epochs_root_file=cfg.epochs_root_file,
                           rois_dict={"lpfc": rois_dict["lpfc"]},
                           electrodes="all")
    keep = resolve_electrodes_to_keep(args, LAB_root)     # {subject: {channels}}
    df = assemble_long_df(subjects_epochs, cfg.window_tmin, cfg.window_tmax,
                          electrodes_to_keep=keep, effect_measure="cohens_d")
    df.attrs["source"] = "real-lpfc"
    return df


def load_synth_df(cfg):
    """Fallback: ground-truth synthetic df with LWPC/LWPS interactions."""
    df = _synthetic_df(effect_measure="cohens_d", seed=cfg.synth_seed)
    # _synthetic_df emits 12 subjects; subset if the user asked for fewer *and* wants that.
    df.attrs["source"] = "synthetic"
    return df


def load_sandbox_df(cfg):
    try:
        df = load_real_lpfc_df(cfg)
        print(f"✓ loaded REAL LPFC data: {len(df)} rows | "
              f"{df.subject.nunique()} subjects | {df.electrode.nunique()} electrodes")
        return df
    except Exception as e:
        print(f"⚠ real load unavailable ({type(e).__name__}: {e})")
        print("  → falling back to SYNTHETIC ground-truth data.")
        df = load_synth_df(cfg)
        print(f"✓ synthetic: {len(df)} rows | {df.subject.nunique()} subjects | "
              f"{df.electrode.nunique()} electrodes")
        return df


df = load_sandbox_df(CONFIG)
print("data source:", df.attrs.get("source"))
df.head()

#### Sanity-look at the data you'll be analyzing
The proportion columns (`incongruent_proportion`, `switch_proportion`) are what make A1 an
*interaction* problem — note their deliberately unequal cell counts (~75/25), the exact
imbalance that makes Type III + sum coding necessary.

In [ ]:
print("columns:", list(df.columns))
print("\nsubjects:", sorted(df.subject.unique()))
print("electrodes per subject:")
print(df.groupby("subject")["electrode"].nunique())
print("\ncongruency × incongruent_proportion cell counts (per-trial rows, one electrode):")
one = df[df.electrode == df.electrode.iloc[0]]
display(pd.crosstab(one.congruency, one.incongruent_proportion))
print("switchType × switch_proportion:")
display(pd.crosstab(one.switchType, one.switch_proportion))

---
# A1 · Per-electrode two-way ANOVA electrode definition

**Goal.** Produce the *parametric, headline* electrode definition: for each electrode, the
**interaction** F/p of a two-way ANOVA — LWPC = `congruency × incongruent_proportion`,
LWPS = `switchType × switch_proportion` — then FDR across electrodes to set the `S`
(stability) and `F` (flexibility) flags. Output columns match `per_electrode_labels` so it's
a drop-in `labels` for `cmh_conjunction`.

**The one subtle thing:** Type III SS with **sum/effect coding**, or a pure main effect
leaks into the "interaction" under the unequal proportion cells. That's the whole
assignment.

### Task A1.1 — `_anova_interaction_stats` (one electrode)
Implement the stub. Then run the check cell.
*Hints:* `reveal("a1_stats_hint1")`, `reveal("a1_stats_hint2")`. *Answer:* `reveal("a1_stats_solution")` or `load_reference("a1_stats")`.

In [ ]:
def _anova_interaction_stats(elec_df, cond_col, mod_col, hg_col="hg"):
    """Fit ONE electrode's two-way Type III ANOVA; return {'F':..., 'p':...} for
    the interaction term (NaN,NaN on a singular fit).

    Steps: sum-coded formula  ->  smf.ols(...).fit()  ->  anova_lm(model, typ=3)
           ->  read the interaction row's 'F' and 'PR(>F)'  (try/except -> NaN)."""
    raise NotImplementedError("A1.1 — implement the per-electrode Type III ANOVA fit")

In [ ]:
# --- quick check on a single electrode -------------------------------------
_g = df[df.electrode == df.electrode.iloc[0]]
try:
    _st = _anova_interaction_stats(_g, "congruency", "incongruent_proportion")
    _fl = _anova_interaction_stats(_g, "switchType", "switch_proportion")
    print("stability (LWPC) interaction:", _st)
    print("flexibility (LWPS) interaction:", _fl)
    assert set(_st) == {"F", "p"}, "return a dict with keys F and p"
    print("✓ shape looks right")
except NotImplementedError as e:
    print("stub not implemented yet:", e)
    print("→ try it, or run:  reveal('a1_stats_hint1')  /  load_reference('a1_stats')")

### Task A1.2 — `per_electrode_anova_labels` (all electrodes → S/F table)
Loop electrodes, get interaction F/p for both processes, extract the **sign** from the
module's own `_interaction_effect` (so it matches §2), FDR across electrodes, set `S`/`F`,
and add the two **cross-interaction controls**.
*Hints:* `reveal("a1_labels_hint1")`, `reveal("a1_labels_hint2")`. *Answer:* `reveal("a1_labels_solution")` / `load_reference("a1_labels")`.

In [ ]:
def per_electrode_anova_labels(df, alpha=0.05, contrast_mode="proportion",
                               contrasts=None, include_cross_controls=True,
                               require_sign=False):
    """Parametric per-electrode S/F labels from the two-way interaction ANOVA.

    Return one row per electrode with at least:
        subject, electrode, p_cong, q_cong, S, p_switch, q_switch, F,
        s_sign, f_sign  (+ p_cross_cs, p_cross_si if include_cross_controls).

    Steps:
      1. contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))
         work = _canonical_labels(df, contrasts)      # attaches _scond/_smod/_fcond/_fmod
      2. per (subject, electrode): _anova_interaction_stats twice; sign via
         np.sign(_interaction_effect(hg, cond, mod, 'cohens_d', alpha)); cross controls.
      3. FDR (fdr_bh) p_cong/p_switch -> q; S=(q_cong<alpha), F=(q_switch<alpha)."""
    raise NotImplementedError("A1.2 — assemble per-electrode ANOVA labels + FDR")

#### Acceptance check A1
The assignment's success criteria, made concrete:
1. **Column compatibility** — `cmh_conjunction(anova_labels)` runs unchanged.
2. **Agreement** with the nonparametric `per_electrode_labels` (same balanced interaction,
   different estimator) — high on synthetic ground truth (Cohen's κ well above 0).
3. **Cross-controls near-null** — no true cross-effect in the generator, so `p_cross_*` is
   ~uniform (rejection rate near α).

In [ ]:
# Build the ANOVA labels, then validate.
anova_labels = per_electrode_anova_labels(df, contrast_mode="proportion")
display(anova_labels.head())

# 1. drop-in compatibility with the existing CMH
conj = cmh_conjunction(anova_labels)
print(f"[1] cmh_conjunction accepts ANOVA labels — MH OR = {conj['mh_odds_ratio']:.3f}")

# 2. agreement vs the nonparametric labels (small n_perm for speed)
nonpar = per_electrode_labels(df, n_perm=200, contrast_mode="proportion")
m = anova_labels.merge(nonpar[["electrode", "S", "F"]], on="electrode",
                       suffixes=("_anova", "_nonpar"))
def _kappa(a, b):
    po = (a == b).mean()
    pe = a.mean()*b.mean() + (1-a.mean())*(1-b.mean())
    return (po - pe) / (1 - pe) if (1 - pe) > 0 else np.nan
for col in ["S", "F"]:
    a, b = m[f"{col}_anova"], m[f"{col}_nonpar"]
    print(f"[2] {col}: agreement={(a==b).mean():.3f}  kappa={_kappa(a,b):+.3f}  "
          f"(anova n={int(a.sum())}, nonpar n={int(b.sum())})")

# 3. cross-control specificity (report only)
if "p_cross_cs" in anova_labels:
    for c in ["p_cross_cs", "p_cross_si"]:
        p = anova_labels[c].dropna()
        print(f"[3] {c}: mean={p.mean():.3f}  frac<0.05={np.mean(p<0.05):.3f}  "
              f"(want ~uniform / ~0.05)")

In [ ]:
# --- visualize the ANOVA labels --------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sc = ax[0].scatter(anova_labels["q_cong"], anova_labels["q_switch"],
                   c=anova_labels["S"] + 2*anova_labels["F"], cmap="viridis",
                   s=22, alpha=.75)
ax[0].axvline(0.05, ls="--", c="k", lw=1); ax[0].axhline(0.05, ls="--", c="k", lw=1)
ax[0].set(xlabel="q (stability / LWPC)", ylabel="q (flexibility / LWPS)",
          title="FDR q-values (dashed = α)", xlim=(-.02, 1.02), ylim=(-.02, 1.02))

both = int(((anova_labels.S==1)&(anova_labels.F==1)).sum())
so   = int(((anova_labels.S==1)&(anova_labels.F==0)).sum())
fo   = int(((anova_labels.S==0)&(anova_labels.F==1)).sum())
ne   = int(((anova_labels.S==0)&(anova_labels.F==0)).sum())
ax[1].bar(["neither","S only","F only","both"], [ne, so, fo, both],
          color=["#ccc", "#2c7fb8", "#d95f0e", "#31a354"])
ax[1].set(title="Electrode selectivity classes (ANOVA)", ylabel="# electrodes")
fig.tight_layout(); plt.show()

---
# A2 · Conjunction permutation null + threshold sweep

`cmh_conjunction` already gives the MH odds ratio and its parametric tests. A2 adds two
robustness pieces the plan asks for:
- a **within-subject permutation null** on the "both" count (respects subject nesting
  better than the analytic hypergeometric), and
- a **threshold sweep** so a segregation claim is shown stable across α, not an artifact of
  one cutoff.

Both are thin wrappers — **reuse** `cmh_conjunction`, don't reimplement the CMH.

### Task A2.1 — `conjunction_permutation_null`
Shuffle `F` **within each subject** (S fixed), recount the overlap, build the null.
*Hints:* `reveal("a2_null_hint1")`, `reveal("a2_null_hint2")`. *Answer:* `reveal("a2_null_solution")` / `load_reference("a2_null")`.

In [ ]:
def conjunction_permutation_null(labels, n_perm=10000, seed=0):
    """Empirical null for the count of 'both' (S==1 & F==1) electrodes.

    Return dict(observed:int, null:ndarray(n_perm), p_two_sided:float, z:float).
    Shuffle F WITHIN each subject only (groupby subject), so each subject's S.sum()
    and F.sum() stay fixed and only the pairing is randomized (the CMH null)."""
    raise NotImplementedError("A2.1 — within-subject permutation null on the overlap count")

#### Acceptance check A2.1
- **Invariant:** every permutation preserves each subject's `S.sum()` and `F.sum()`
  (only the pairing moves). We assert it directly on the null machinery.
- **Behavior on independent data:** the synthetic generator draws each electrode's
  stability/flexibility sensitivities independently, so the observed overlap should sit
  *inside* the null (p not significant, z near 0).

In [ ]:
null_res = conjunction_permutation_null(anova_labels, n_perm=3000, seed=1)
print(f"observed 'both' = {null_res['observed']} | "
      f"null mean = {null_res['null'].mean():.2f} | "
      f"p(two-sided) = {null_res['p_two_sided']:.3f} | z = {null_res['z']:+.2f}")

# --- assert the within-subject invariant directly --------------------------
_lab = anova_labels.dropna(subset=['S','F']).copy()
_S = _lab['S'].to_numpy(); _F = _lab['F'].to_numpy(); _subj = _lab['subject'].to_numpy()
_groups = [np.where(_subj==s)[0] for s in np.unique(_subj)]
_rng = np.random.default_rng(0)
_S_by_subj = _lab.groupby('subject')['S'].sum()
_F_by_subj = _lab.groupby('subject')['F'].sum()
for _ in range(50):
    _Fp = _F.copy()
    for idx in _groups:
        _Fp[idx] = _F[_rng.permutation(idx)]
    perm = _lab.assign(Fp=_Fp)
    assert (perm.groupby('subject')['S'].sum() == _S_by_subj).all()
    assert (perm.groupby('subject')['Fp'].sum() == _F_by_subj).all()
print("✓ invariant holds: per-subject S and F counts unchanged under permutation")

In [ ]:
# --- visualize the null ----------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_res["null"], bins=30, color="#bbb", edgecolor="w")
ax.axvline(null_res["observed"], color="#d7191c", lw=2.5,
           label=f"observed = {null_res['observed']}")
ax.set(title=f"Within-subject permutation null for 'both' count "
             f"(p={null_res['p_two_sided']:.3f})",
       xlabel="# 'both' electrodes under null", ylabel="# permutations")
ax.legend(); fig.tight_layout(); plt.show()

### Task A2.2 — `conjunction_threshold_sweep`
Recompute the counts + MH OR across a range of selection thresholds (here, q-value cutoffs
on the ANOVA labels). Loop, call `cmh_conjunction`, return a tidy table.
*Hint:* `reveal("a2_sweep_hint1")`. *Answer:* `reveal("a2_sweep_solution")` / `load_reference("a2_sweep")`.

In [ ]:
def conjunction_threshold_sweep(labels_by_threshold, thresholds):
    """Recompute overlap OR + counts across selection thresholds.

    labels_by_threshold : callable  threshold -> labels DataFrame (S/F recomputed).
    Return a tidy DataFrame: threshold, n_S, n_F, n_both, mh_odds_ratio, cmh_p."""
    raise NotImplementedError("A2.2 — threshold sweep over the conjunction OR")

#### Acceptance check A2.2
On the independent synthetic data the OR should **hover near 1 across the whole sweep** and
the conclusion (segregated vs. shared) should not flip at any reasonable threshold. On real
LPFC data, a sweep that stays on one side of 1 is the robustness evidence the plan wants;
one that flips is a *finding to report, not hide*.

In [ ]:
# Threshold on the ANOVA q-values: S/F = (q < cutoff).
def labels_at_qcut(qcut):
    l = anova_labels.copy()
    l["S"] = (l["q_cong"]   < qcut).astype(int)
    l["F"] = (l["q_switch"] < qcut).astype(int)
    return l

thresholds = [0.01, 0.05, 0.10, 0.20, 0.35, 0.50]
sweep = conjunction_threshold_sweep(labels_at_qcut, thresholds)
display(sweep)

# cross-check: the continuous correlation should tell the same story at the sweep midpoint
elec = add_responsiveness(
    compute_sensitivities(df, n_splits=30, contrast_mode="proportion"), df)
cont = prepare_continuous(elec)
corr = subject_clustered_corr(cont, n_perm=1000)
print(f"\ncontinuous corr = {corr['corr']:+.3f} (p={corr['p']:.3f})  "
      f"[>0 shared core, ≤0 segregated] — compare to OR vs 1 in the sweep")

In [ ]:
# --- plot OR vs threshold ---------------------------------------------------
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(sweep["threshold"], sweep["mh_odds_ratio"], "o-", color="#31a354", lw=2,
         label="MH odds ratio")
ax1.axhline(1.0, ls="--", c="k", lw=1, label="OR = 1 (independence)")
ax1.set(xlabel="selection threshold (q-value cutoff)", ylabel="MH odds ratio")
ax1.set_title("Conjunction OR across thresholds\n(OR<1 → segregation, OR>1 → shared core)")
ax2 = ax1.twinx()
ax2.bar(sweep["threshold"], sweep["n_both"], width=0.015, alpha=.25, color="#2c7fb8")
ax2.set_ylabel("# 'both' electrodes", color="#2c7fb8")
ax1.legend(loc="upper left"); fig.tight_layout(); plt.show()

---
## Where this code goes next

You now have working, tested versions of all four functions. To promote them:

1. Paste `_anova_interaction_stats` + `per_electrode_anova_labels` into
   `src/analysis/stats/stability_flexibility_segregation.py` (next to
   `per_electrode_labels`), and `conjunction_permutation_null` +
   `conjunction_threshold_sweep` next to `cmh_conjunction`.
2. Delete the corresponding `docs/skeletons/a1_*.py` / `a2_*.py` stubs.
3. Add the acceptance checks above as unit tests under `tests/` (they already encode the
   assignment's criteria: κ-agreement, near-null cross-controls, the permutation
   invariant, OR≈1 on independent data).

### Keep the §0 cross-cutting principles in view
- **Double-dipping / disjoint halves** — the sign here comes from `_interaction_effect`,
  which the pipeline estimates on disjoint trial halves in `compute_sensitivities`.
- **Power matching** — that's exactly what the A2 sweep is for: report OR *as a function of*
  threshold, never one α snapshot.
- **FDR** across electrodes — baked into `per_electrode_anova_labels`.
- **Coverage bias, latency–amplitude, decoding confounds** — belong to A3/A4/A5; not
  touched here, but don't forget them when you extend.

**Next assignments:** A3 (anatomy), A4 (cross-decoding), A5 (timing), A6 (brain–behavior) —
skeletons in `docs/skeletons/`.